Live Video Capture

In [ ]:
!pip install gradio deepface cryptography opencv-python mediapipe

In [2]:
import cv2

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (640, 640))  # Normalize size
    cv2.imwrite(image_path, img)

In [ ]:
pip install gradio

In [ ]:
pip install deepface

In [ ]:
pip install --upgrade pip

In [10]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install deepface opencv-python mediapipe cryptography gradio numpy

In [1]:
import gradio as gr
import os
import json
import numpy as np
import cv2
import mediapipe as mp
from deepface import DeepFace
from cryptography.fernet import Fernet
from numpy import dot
from numpy.linalg import norm


DB_FILE = "users_secure.json"

if not os.path.exists(DB_FILE):
    with open(DB_FILE, "w") as f:
        json.dump({}, f)

if not os.path.exists("secret.key"):
    key = Fernet.generate_key()
    with open("secret.key", "wb") as f:
        f.write(key)

with open("secret.key", "rb") as f:
    key = f.read()

cipher = Fernet(key)

def load_users():
    with open(DB_FILE, "r") as f:
        return json.load(f)

def save_users(users):
    with open(DB_FILE, "w") as f:
        json.dump(users, f)

def generate_embedding(image_path):
    preprocess_image(image_path)

    embedding = DeepFace.represent(
        img_path=image_path,
        model_name="ArcFace",
        detector_backend="mtcnn",
        enforce_detection=False
    )[0]["embedding"]

    return np.array(embedding)

def average_embeddings(embeddings):
    return np.mean(embeddings, axis=0)

def cosine_similarity(e1, e2):
    return dot(e1, e2) / (norm(e1) * norm(e2))


def liveness_check(image_path):
    try:
        DeepFace.extract_faces(
            img_path=image_path,
            detector_backend="mtcnn",
            enforce_detection=True
        )
        return True
    except:
        return False

def register_user(user_id, img1, img2, img3):
    users = load_users()

    if user_id in users:
        return "❌ User already exists"

    images = [img1, img2, img3]
    embeddings = []

    for i, img in enumerate(images):
        if img is None:
            return "❌ Upload all 3 images (Front, Left, Right)"

        path = f"reg_{i}.jpg"
        img.save(path)

        if not liveness_check(path):
            return "❌ Face not detected properly"

        emb = generate_embedding(path)
        embeddings.append(emb)

    avg_emb = average_embeddings(embeddings)

    encrypted = cipher.encrypt(json.dumps(avg_emb.tolist()).encode())

    users[user_id] = {
        "embedding": encrypted.decode(),
        "voted": False
    }

    save_users(users)

    return "✅ Multi-angle Registration Successful"

def login_and_vote(user_id, image):
    users = load_users()

    if user_id not in users:
        return "❌ User not found"

    path = "login.jpg"
    image.save(path)

    if not liveness_check(path):
        return "❌ Liveness Failed"

    new_emb = generate_embedding(path)

    encrypted = users[user_id]["embedding"].encode()
    stored_emb = np.array(json.loads(cipher.decrypt(encrypted).decode()))

    similarity = cosine_similarity(stored_emb, new_emb)

    if similarity > 0.75:
        if users[user_id]["voted"]:
            return f"✅ Login Successful (Already Voted) | Score: {similarity:.3f}"
        else:
            users[user_id]["voted"] = True
            save_users(users)
            return f"🗳 Vote Cast Successfully! | Score: {similarity:.3f}"
    else:
        return f"❌ Face Not Matched | Score: {similarity:.3f}"

with gr.Blocks() as app:
    gr.Markdown("# 🔐 Secure Face Voting System")

    with gr.Tab("Register (3 Angles)"):
        reg_id = gr.Textbox(label="User ID")
        img1 = gr.Image(type="pil", label="Front Face")
        img2 = gr.Image(type="pil", label="Left Angle")
        img3 = gr.Image(type="pil", label="Right Angle")
        reg_btn = gr.Button("Register")
        reg_output = gr.Textbox()

        reg_btn.click(register_user, inputs=[reg_id, img1, img2, img3], outputs=reg_output)

    with gr.Tab("Login & Vote"):
        login_id = gr.Textbox(label="User ID")
        login_img = gr.Image(type="pil", label="Live Face")
        login_btn = gr.Button("Login & Vote")
        login_output = gr.Textbox()

        login_btn.click(login_and_vote, inputs=[login_id, login_img], outputs=login_output)

app.launch(share=True)

c:\Users\Abhishek\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'deepface'